## Getting Started with Anthropic API

Introduction & Goals

Welcome to your first lesson in building effective agents with Claude! Whether you're completely new to the Anthropic API or have some experience with it, this lesson will ensure you have a solid foundation for the advanced agent-building techniques we'll cover later in the path.

In this lesson, you'll learn how to send messages to Claude using the Anthropic API and understand the complete response structure. By the end, you'll be able to create a Python script that communicates with Claude and examine the full JSON response. This understanding is crucial because throughout this course, we'll be working with different parts of Claude's responses — from basic text content to tool usage metadata and conversation flow control.

This foundation is essential because later lessons will extend this same pattern to develop more complex workflows with Claude.

Environment and Setup

To communicate with Claude, you'll need two things: the Anthropic Python SDK and an API key from Anthropic. The SDK handles all the technical details of making API requests, and you'd normally install it using `pip install anthropic`. The API key authenticates your requests, and the Anthropic client automatically looks for it in the `ANTHROPIC_API_KEY` environment variable.

In CodeSignal, we've already configured everything for you — the SDK is pre-installed and your API key is set up, so you can focus on learning the core concepts without worrying about setup details.

How Claude Messaging Works

Every interaction with Claude follows a structured conversation pattern. Understanding this structure is key to building effective agents, as you'll need to manage conversation state and interpret various response components throughout this course.

Claude recognizes three conversation roles. The `system` role sets the context and instructions for Claude's behavior — essentially Claude's job description for the conversation. The `user` role represents messages from you or your application users. The `assistant` role represents Claude's responses. This role-based system helps Claude maintain context and understand conversation flow, which becomes critical when building multi-step agent workflows.

When you send a request to Claude, you package several pieces of information: the model you want to use, a system prompt that defines Claude's behavior, an array of messages representing the conversation history, and a `max_tokens` limit for the response length. Tokens roughly correspond to words, so `max_tokens: 2000` allows Claude to respond with approximately 1,500–2,000 words.

The request flows to Anthropic's servers where Claude processes your messages and generates a response. That response returns as a structured JSON object containing Claude's message plus metadata about the interaction — information we'll use extensively in later lessons for tool usage tracking, conversation management, and error handling.

Setting Up the Client and Configuration

Let's build our first Claude interaction by examining each component. We'll start with the basic imports and client initialization, then define our model and system prompt:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."
```

The client automatically finds your API key in the environment variables. The system prompt influences how Claude responds throughout the conversation — it's like setting Claude's personality and expertise for the entire interaction. Understanding system prompts is crucial because later in the course, we'll use them to define how Claude should use tools and handle complex agent workflows.

Creating Your First Message

Now we'll create the messages array representing our conversation:

```python
# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]
```

Each message is a dictionary with `role` and `content` fields. Even for a single message, we use an array because conversations can have multiple exchanges.

Sending the Message to Claude

With our message prepared, we can now send it to Claude:

```python
# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)
```

The `client.messages.create()` method sends an HTTP request to Anthropic's servers, where Claude processes your message according to the system prompt and returns a structured response.

Note that `max_tokens` is a required parameter that limits how long Claude's response can be. Think of Claude as having two token limits: a context window (how much total conversation history it can remember) and a response limit (how much it can write back to you). The context window for Claude Sonnet 4.6 is 1 million tokens, which can hold roughly 750,000 words of conversation history. The `max_tokens` parameter controls the response limit — setting it to 2000 means Claude can respond with up to about 1,500-2,000 words, leaving the rest of the context window available for your conversation history.

Examining the Complete Response Structure

To understand what Claude returns, let's examine the complete response structure:

```python
import json

# Print the whole response as JSON
print(json.dumps(response.model_dump(), indent=2))
```

The `response.model_dump()` method converts Claude's response into a Python dictionary, and `json.dumps()` formats it as readable JSON. You'll see output like this:

```json
{
  "id": "msg_01QPRnXZFU9NHQYsw9dujcNJ",
  "content": [
    {
      "citations": null,
      "text": "The main difference is that dogs are typically more social and require active engagement (walks, training, interaction), while cats are more independent and low-maintenance, often content with minimal daily care.",
      "type": "text"
    }
  ],
  "model": "claude-sonnet-4-6",
  "role": "assistant",
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "input_tokens": 29,
    "output_tokens": 41,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}
```

This JSON structure contains everything you need to understand how Claude processed your request and what it returned.

Understanding Response Fields

Understanding this response structure is essential for the rest of the course. Key fields include:

- `id` - Provides a unique identifier useful for logging and debugging
- `content` - Contains Claude's response as an array of content blocks. Notice it's an array because responses can contain multiple blocks of different types — text blocks like we see here, but also thinking blocks, tool usage blocks, and other content types we'll explore later
- `stop_reason` - Tells you why Claude stopped generating text. "end_turn" means Claude naturally concluded its response, but you'll encounter other values like "tool_use" in later lessons when Claude decides to call a function
- `usage` - Provides detailed token consumption information, which becomes important for monitoring agent performance and costs

Pay special attention to the content array structure. Each block has a `type` field (here it's "text") and the actual content.

Extracting the Text Response

Most of the time, you'll want to access just Claude's text response rather than the full JSON structure. Let's see how to extract the clean text content:

```python
# Print the text response
print(response.content[0].text)
```

Since our response contains only one content block, we can access it directly with `response.content[0].text`. This is the easiest approach for simple interactions, but it won't always be this straightforward — later in the course, you'll encounter responses with multiple content blocks of different types that require more sophisticated handling.

This produces the clean text output:

```
The main difference is that dogs are typically more social and require active engagement (walks, training, interaction), while cats are more independent and low-maintenance, often content with minimal daily care.
```


When we access `response.content[0].text`, we're getting the text from the first content block. This understanding of content block filtering will be crucial as we progress to more complex agent workflows.

Building Multi-Turn Conversations

Real conversations don't end after one exchange. To continue our conversation with Claude, we need to maintain the conversation history by adding Claude's response to our messages array, then append our follow-up question:

```python
# Append Claude's response to messages
messages.append({"role": "assistant", "content": response.content})

# Append a new user message
messages.append({"role": "user", "content": "Which one is easier to train?"})

# Send the second request
second_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)
```

Notice how we append Claude's entire content array to maintain the conversation structure. This preserves all content blocks and their types, which becomes crucial when working with responses that contain multiple content types. Now let's see Claude's response to our follow-up question:

```python
# Print the text response
print(second_response.content[0].text)
```

This produces output like:
    Dogs are generally easier to train. They're naturally more eager to please their owners and respond well to commands and rewards, while cats are more independent and selective about following instructions.


The conversation continues naturally because Claude can see the full context of our previous exchange, allowing it to provide a focused answer about training specifically.

Enabling Extended Thinking

Claude can also show its reasoning process through "thinking" — internal deliberation that helps it provide better responses. Let's continue our conversation with thinking enabled to see how Claude works through problems:

```python
# Append Claude's second response to messages
messages.append({"role": "assistant", "content": second_response.content})

# Append a third user message
messages.append({"role": "user", "content": "What about grooming requirements?"})

# Send the third request with thinking enabled
third_response = client.messages.create(
    model=model,
    max_tokens=16000,
    messages=messages,
    system=system_prompt,
    thinking={
        "type": "enabled",
        "budget_tokens": 10000
    }
)
```

When thinking is enabled, Claude's response can contain multiple content blocks of different types.

Examining Thinking Responses

Let's examine the full response structure to see both Claude's internal reasoning and its final answer:

```python
# Print the whole response as JSON first
print(json.dumps(third_response.model_dump(), indent=2))
```

You'll see output that includes both a thinking block (Claude's internal reasoning) and a text block (the final answer):

```json
{
  "id": "msg_01NBMAsNFeiDzYP2ue1rXptY",
  "content": [
    {
      "signature": "Ep8GCkYIBxgCKkDAB43...",
      "thinking": "The user is asking about grooming requirements for cats vs dogs. Let me think about this:\n\nDogs:\n- Need regular brushing depending on coat type (some daily, some weekly)\n- Need regular baths (every few weeks to months depending on breed/lifestyle)\n- Need nail trimming\n- Need ear cleaning\n- Professional grooming may be needed for certain breeds\n\nCats:\n- Generally groom themselves very well\n- May need occasional brushing (especially long-haired breeds)\n- Rarely need baths unless they get into something messy\n- Need nail trimming\n- Generally lower maintenance for grooming\n\nSo cats are generally lower maintenance for grooming.",
      "type": "thinking"
    },
    {
      "citations": null,
      "text": "Cats have lower grooming requirements. They groom themselves naturally and rarely need baths. Dogs typically need regular brushing, occasional baths, and some breeds require professional grooming. However, both need nail trims and basic care.",
      "type": "text"
    }
  ],
  "model": "claude-sonnet-4-6",
  "role": "assistant",
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "input_tokens": 156,
    "output_tokens": 206,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}
```

Notice how the response now contains two content blocks: one showing Claude's internal reasoning process and another with the polished final answer.

Processing Multiple Content Blocks

When Claude's response contains multiple content blocks (like thinking and text), we need a way to extract just the parts we want. Let's filter the response to get only the text content that we'd show to a user:

```python
# Get all text content from the response
text_contents = [block.text for block in third_response.content if block.type == "text"]

# Join all text contents and print them
print("\n".join(text_contents))
```

This code loops through all content blocks in the response, finds only the ones with `type == "text"`, extracts their text, and joins them together. This produces the clean final output:

    Cats have lower grooming requirements. They groom themselves naturally and rarely need baths. Dogs typically need regular brushing, occasional baths, and some breeds require professional grooming. However, both need nail trims and basic care


This pattern of filtering content blocks by type is essential for building robust agents. Later in the course, you'll encounter responses with tool usage blocks, multiple text blocks, and other content types that require similar filtering and processing techniques.

Summary & Next Steps

You've now successfully understood how to interact with Claude using the Anthropic API, examined the complete response structure, built multi-turn conversations, and explored extended thinking capabilities. You understand how to structure conversations with roles, send requests with system prompts, maintain conversation context, and interpret response metadata including different content block types.

The key concepts you've learned — conversation management, content block filtering, and response structure analysis — form the foundation for the advanced agent-building techniques we'll cover throughout this course.

In the upcoming practices, you'll get hands-on experience building on these concepts and exploring different ways to interact with Claude. This foundation will serve you well as we progress through more advanced topics in the course!




## Making Your First Claude Request

Now that you've learned about the core components of Claude interactions, it's time to build your first working conversation with the API! Your goal is to create a complete Claude interaction by filling in the missing pieces of code that will send a message to Claude and display the full response structure.

You'll need to complete several key steps to make this work:

    Initialize the Anthropic client to connect to the API.
    Set up the model name to "claude-sonnet-4-6" and create a system prompt.
    Build a messages array with a user question.
    Send the request to Claude with all the required parameters.

Once you complete this task, the code should display Claude's complete response as formatted JSON, letting you see the full structure that you'll work with throughout this course. This hands-on practice will give you confidence with the fundamental building blocks of every Claude interaction!

```
import json
from anthropic import Anthropic

# TODO: Initialize the Anthropic client
client = 

# TODO: Define the model to use
model = 

# TODO: Create a short system prompt that makes Claude a helpful assistant who answers briefly
system_prompt = 

# TODO: Create an array of messages to send to Claude with a single user message
messages = 

# TODO: Send the messages to Claude using client.messages.create() with your model, system prompt and max tokens
response = 

# Print the whole response as JSON
print(json.dumps(response.model_dump(), indent=2))
```

To complete this task and make your first successful API request to Claude, you need to use the official Anthropic Python SDK.

Here is the complete, working code with all the missing pieces filled in correctly:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
# By default, it automatically reads the ANTHROPIC_API_KEY environment variable.
client = Anthropic()

# Define the model to use
model = "claude-sonnet-4-6"

# Create a short system prompt that makes Claude a helpful assistant who answers briefly
system_prompt = "You are a helpful assistant. Keep all your answers very brief and to the point."

# Create an array of messages to send to Claude with a single user message
messages = [
    {
        "role": "user",
        "content": "What is the distance between the Earth and the Moon in kilometers?"
    }
]

# Send the messages to Claude using client.messages.create() with your model, system prompt, and max tokens
response = client.messages.create(
    model=model,
    system=system_prompt,
    messages=messages,
    max_tokens=1000  # Sets a safe ceiling limit for the generation tokens
)

# Print the whole response as JSON
print(json.dumps(response.model_dump(), indent=2))

```

### 📝 Key Step Explanations

1. **`Anthropic()`**: This initializes the connection client. Make sure you have your API key set up in your terminal environment as an environment variable before running the script:
```bash
export ANTHROPIC_API_KEY="your-actual-api-key"

```



```
2. **`system`**: The system prompt is passed directly as a top-level parameter (`system=system_prompt`) in the SDK, separate from the `messages` array.
3. **`messages`**: This is structured as a list of dictionaries. Each message must contain a `"role"` (which can be `"user"` or `"assistant"`) and `"content"`.
4. **`max_tokens`**: This parameter is mandatory for every request to define the maximum generation tokens allowed for Claude's response.

```

## Extracting Key Information from Responses

Great work on setting up your first Claude conversation! Now it's time to move beyond examining the raw JSON response and learn how to extract the specific information you'll actually use in your applications.

Instead of printing the entire response structure, you'll focus on pulling out the two most important pieces of information from Claude's reply. Your task is to extract and display Claude's actual text response and the stop_reason that tells you why Claude finished generating its answer.

You'll need to:

    Extract the text content from the first content block in the response.
    Extract the stop_reason field from the response.
    Print both pieces of information with clear, descriptive labels.

This exercise teaches you the essential skill of navigating Claude's response structure to access the data you need for real applications, rather than working with the full JSON dump.

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)


# TODO: Extract and print the text content from the first content block with the label "Claude's Response:"

# TODO: Extract and print the stop_reason field with the label "Stop Reason:"

```

To complete this task, you need to access the appropriate attributes of the `Message` object returned by the Anthropic SDK.

* **Claude's Response Text:** The generated content is stored as a list of content blocks in the `content` attribute (`response.content`). For standard text generation, the text string can be found in the first block under the `text` field: `response.content[0].text`.
* **Stop Reason:** The top-level `stop_reason` attribute tells you why the generation ended (e.g., `"end_turn"` if it finished naturally, or `"max_tokens"` if it hit the limit): `response.stop_reason`.

Here is the complete, working code with the extraction steps implemented:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)


# Extract and print the text content from the first content block with the label "Claude's Response:"
claude_response_text = response.content[0].text
print(f"Claude's Response: {claude_response_text}")

# Extract and print the stop_reason field with the label "Stop Reason:"
generation_stop_reason = response.stop_reason
print(f"Stop Reason: {generation_stop_reason}")

```

## Building Multi Turn Conversations with Claude

Building on your success in extracting key information from Claude's responses, it's time to unlock the real power of conversational AI by creating a multi-turn dialogue! Your mission is to transform a single-question interaction into a flowing conversation in which Claude can build upon previous context.

You'll start with a working foundation that asks Claude about the differences between cats and dogs as pets. From there, you need to extend this into a two-part conversation by implementing the essential pattern that all conversational applications use.

Your tasks are to:

    Add Claude's first response to your messages array using the correct assistant role
    Include a follow-up user question asking, "Which one is easier to train?"
    Send this expanded conversation back to Claude
    Display Claude's contextual response to your follow-up question


```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the text response
print(response.content[0].text)

# TODO: Append Claude's response to messages with the assistant role

# TODO: Append a new user message asking "Which one is easier to train?"

# TODO: Send the second request to Claude using the same parameters as the first request

# TODO: Print the text response from the second interaction
```

To build a multi-turn conversation with Claude, you must append both Claude's response (as the `assistant` role) and your new question (as the `user` role) to the `messages` array before sending the next request. This allows the model to maintain the full context of the dialogue.

Here is the complete, working Python code with the multi-turn logic implemented:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the first request to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the first text response
print(response.content[0].text)

# Append Claude's response to messages with the assistant role
messages.append({
    "role": "assistant",
    "content": response.content[0].text
})

# Append a new user message asking "Which one is easier to train?"
messages.append({
    "role": "user",
    "content": "Which one is easier to train?"
})

# Send the second request to Claude using the same parameters as the first request
follow_up_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the text response from the second interaction
print(follow_up_response.content[0].text)

```

### 💡 Core Architecture Concept

Anthropic's API is stateless, meaning it does not remember past interactions on its own. To create a conversational flow, your application code must explicitly act as the memory bank by passing the cumulative message history (`user` $\rightarrow$ `assistant` $\rightarrow$ `user`) back to Claude with each new turn.

## Handling Multiple Content Blocks Safely

Excellent work mastering multi-turn conversations! Now it's time to make your code more robust by learning how to handle responses that might contain multiple content blocks of different types.

Currently, your conversation code uses the simple approach of accessing response.content[0].text to get Claude's answer. While this works fine for basic responses, it can break when Claude's responses contain multiple content blocks or different content types. You need to upgrade your text extraction method to be more flexible and reliable.

Your task is to replace the simple text extraction in the second print statement with a more sophisticated approach:

    Use a list comprehension to filter through all content blocks in the second_response.
    Select only the blocks that have type == "text".
    Extract the text content from these filtered blocks.
    Join all the text contents together using "\n".join().

This pattern will serve you well as you progress to more advanced Claude features, where responses can contain thinking blocks, tool usage blocks, and other content types alongside the main text response!

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the text response
print(response.content[0].text)

# Append Claude's response to messages
messages.append({"role": "assistant", "content": response.content})

# Append a new user message
messages.append({"role": "user", "content": "Which one is easier to train?"})

# Send the second request
second_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# TODO: Replace the simple text extraction with a robust approach that filters all content blocks for text type and joins them
print(second_response.content[0].text)

```

To handle multi-turn conversations safely and prevent your code from crashing when Claude returns modern, multimodal, or structured content types (like thinking blocks or tool calls), filtering the content array is the best practice.

Here is the complete, robust implementation using a list comprehension to safely extract and join all text blocks from the `second_response`:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt starting with "You are"
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the text response
print(response.content[0].text)

# Append Claude's response to messages
messages.append({"role": "assistant", "content": response.content})

# Append a new user message
messages.append({"role": "user", "content": "Which one is easier to train?"})

# Send the second request
second_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Replace the simple text extraction with a robust approach that filters all content blocks for text type and joins them
claude_text_blocks = [block.text for block in second_response.content if block.type == "text"]
full_text_response = "\n".join(claude_text_blocks)

print(full_text_response)

```

### 🧠 Why This Pattern is Crucial

Advanced models frequently return multiple blocks in the `.content` field. For example, if Claude uses a tool or displays internal reasoning, the array structure might look like this:

1. `second_response.content[0]` $\rightarrow$ *Thinking Block* (type: `"thinking"`)
2. `second_response.content[1]` $\rightarrow$ *Text Block* (type: `"text"`)

Using the brittle `content[0].text` approach here would throw an `AttributeError` because a thinking or tool block does not share the same string attributes as a text block. Filtering with `if block.type == "text"` guarantees your application isolates the raw conversational output safely.

## Exploring Claude's Internal Thinking Process

Perfect! You've mastered the art of handling multiple content blocks safely in Claude responses. Now it's time to unlock one of Claude's most powerful features: extended thinking, which lets you peek inside Claude's reasoning process before it gives you the final answer!

Your mission is to modify your second API call to enable Claude's thinking capabilities:

    Add the thinking parameter with type set to "enabled" and budget_tokens set to 10000
    Increase max_tokens to 16000 to accommodate both the thinking content and the final response
    Examine the complete JSON response to see how thinking appears as a separate content block

Once you make these changes, you'll see Claude's internal reasoning process alongside its final answer, demonstrating how different content block types work together in sophisticated AI responses. This advanced capability will become essential as you build more complex conversational agents that need to understand and work with Claude's reasoning process!

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the text response
print("First response text:")
print(response.content[0].text)

# Append Claude's response to messages
messages.append({"role": "assistant", "content": response.content})

# Append a new user message
messages.append({"role": "user", "content": "Which one is easier to train?"})

# TODO: Add thinking parameter with type "enabled" and budget_tokens 10000
# Remember to increase your max_tokens to accomodate all the tokens
second_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the whole response as JSON
print("\nSecond response JSON:")
print(json.dumps(second_response.model_dump(), indent=2))

# Get all text content from the response
print("\nSecond response text:")
text_contents = [block.text for block in second_response.content if block.type == "text"]
print("\n".join(text_contents))
```

To enable Claude's internal thinking process (extended thinking), you need to provide a `thinking` dictionary parameter with `type="enabled"` and a specified `budget_tokens` limit. Additionally, as instructed, you must bump up the `max_tokens` ceiling to `16000` to prevent the response from being cut off prematurely, since both the internal reasoning blocks and the final conversational text share the same token pool.

Here is the complete, working Python code with the extended thinking configuration added to your second API request:

```python
import json
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Short system prompt
system_prompt = "You are a helpful assistant. Answer questions very briefly."

# Create an array of messages to send to Claude
messages = [
    {
        "role": "user",
        "content": "What is the main difference between cats and dogs as pets"
    }
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt
)

# Print the text response
print("First response text:")
print(response.content[0].text)

# Append Claude's response to messages
messages.append({"role": "assistant", "content": response.content})

# Append a new user message
messages.append({"role": "user", "content": "Which one is easier to train?"})

# Add thinking parameter with type "enabled" and budget_tokens 10000
# Remember to increase your max_tokens to accommodate all the tokens
second_response = client.messages.create(
    model=model,
    max_tokens=16000,  # Increased to safely fit both thinking blocks and final text
    messages=messages,
    system=system_prompt,
    thinking={
        "type": "enabled",
        "budget_tokens": 10000
    }
)

# Print the whole response as JSON
print("\nSecond response JSON:")
print(json.dumps(second_response.model_dump(), indent=2))

# Get all text content from the response
print("\nSecond response text:")
text_contents = [block.text for block in second_response.content if block.type == "text"]
print("\n".join(text_contents))

```

### 🧠 What to Look For in the Second Response JSON

When you print the full `second_response` JSON structure, you will see a new block object appear inside the `.content` array before your final text block:

```json
{
  "type": "thinking",
  "thinking": "Dogs are pack animals with an instinctual desire to please a leader, making them highly responsive to operant conditioning..."
}

```

This structure clearly illustrates why our filtering logic (`if block.type == "text"`) is necessary. The `thinking` content block does not have a `.text` attribute (it uses `.thinking`), so isolating blocks specifically by their type ensures your text parsers won't crash when encountering Claude's raw internal chains of thought.